# Index embeddings into embedding store

# Set up

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

import torch
import pandas as pd
from dotenv import load_dotenv
from loguru import logger
from pydantic import BaseModel
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

import mlflow

sys.path.insert(0, "..")

from src.cfg import ConfigLoader

load_dotenv()

True

# Controller

In [3]:
cfg = ConfigLoader("../cfg/common.yaml")
cfg

{
  "run": {
    "author": "",
    "testing": false,
    "log_to_mlflow": true,
    "experiment_name": null,
    "run_name": null,
    "run_persist_dir": null,
    "random_seed": 41
  },
  "data": {
    "hf_datasets": {
      "name": "McAuley-Lab/Amazon-Reviews-2023",
      "mcauley_variant": "Books"
    },
    "train_fp": "/home/dvq/frostmourne/recsys-blog/1-seq-model/data/train.parquet",
    "val_fp": "/home/dvq/frostmourne/recsys-blog/1-seq-model/data/val.parquet",
    "idm_fp": "/home/dvq/frostmourne/recsys-blog/1-seq-model/data/idm.json",
    "metadata_fp": "/home/dvq/frostmourne/recsys-blog/1-seq-model/data/metadata.parquet",
    "train_features_fp": "/home/dvq/frostmourne/recsys-blog/1-seq-model/data/train_features.parquet",
    "val_features_fp": "/home/dvq/frostmourne/recsys-blog/1-seq-model/data/val_features.parquet",
    "full_features_neg_fp": "/home/dvq/frostmourne/recsys-blog/1-seq-model/data/full_features_neg_sampling_df.parquet",
    "train_features_neg_fp": "/home/dvq/

# Load model

In [4]:
mlf_client = mlflow.MlflowClient()

In [5]:
mlf_model = mlflow.pyfunc.load_model(model_uri=f"models:/{cfg.train.retriever.mlf_model_name}@champion")

/home/dvq/frostmourne/recsys-blog/1-seq-model/.venv/lib/python3.11/site-packages/mlflow/pyfunc/utils/data_validation.py:168: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [6]:
run_id = mlf_model.metadata.run_id
run_info = mlf_client.get_run(run_id).info
artifact_uri = run_info.artifact_uri

In [7]:
sample_input = mlflow.artifacts.load_dict(f"{artifact_uri}/inferrer/input_example.json")
sample_input

{'user_ids_raw': ['AE224PFXAEAT66IXX43GRJSWHXCA'],
 'item_seq_raw': [['0007149824', '0007149832']],
 'candidate_items_raw': ['0007149824']}

In [ ]:
prediction = mlf_model(sample_input)
prediction

/home/dvq/frostmourne/recsys-blog/1-seq-model/notebooks/../src/sequence/inference.py:68: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  item_sequences = torch.tensor(item_sequences)


{'user_ids_raw': ['AE224PFXAEAT66IXX43GRJSWHXCA'],
 'item_seq_raw': [['0007149824', '0007149832']],
 'candidate_items_raw': ['0007149824'],
 'scores': [0.5968694686889648]}

# Get embeddings

In [9]:
inferer = mlf_model.unwrap_python_model()
id_mapping = inferer.idm
all_items = list(id_mapping.item_to_index.values())
num_items = len(all_items)
logger.info(f"{all_items[:5]=}, {num_items=:,.0f}")

2025-03-11 20:11:41.751 | INFO     | __main__:<module>:5 - all_items[:5]=[0, 1, 2, 3, 4], num_items=7,388


In [11]:
model = inferer.model
model.eval()

all_item_ids = torch.arange(num_items)
inputs = {
    "candidate_items": all_item_ids
}

with torch.no_grad():
    candidate_embeddings = model.get_candidate_embeddings(inputs).detach().numpy()

embedding_dim = candidate_embeddings.shape[1]
logger.info(f"{embedding_dim=}")
candidate_embeddings[0]

2025-03-11 20:12:39.423 | INFO     | __main__:<module>:13 - embedding_dim=128


array([-0.10703436, -0.17216803, -0.709746  , -0.36154547,  0.0796136 ,
        0.68052334, -0.41148958,  0.6020838 , -0.6856135 , -0.76566297,
        0.80649155,  0.11452065,  0.36899605,  0.5482804 , -0.2100071 ,
        0.7301396 ,  0.21019037,  0.14870211,  0.10973708,  0.12490606,
       -0.81939036,  0.4751542 ,  0.8970341 ,  0.3210783 ,  0.2444923 ,
       -0.45022163, -0.6635965 ,  0.05767518,  0.83844936, -0.2673742 ,
       -0.31369504, -0.8094832 ,  0.8129729 , -0.07270477,  0.01270208,
       -1.3898419 , -0.18922094, -0.16148195,  0.6341016 ,  0.8318451 ,
        0.04122666, -0.59477454, -0.31984943, -0.14195304, -0.8827549 ,
       -0.17894791, -0.10703595,  1.311721  , -1.0553167 , -0.5977145 ,
        1.1542509 ,  0.27556774, -0.589627  ,  0.08037068, -0.07314882,
        0.56439096,  0.64440215,  0.4762257 , -0.11130129,  0.3758156 ,
       -0.24041894, -0.00555591, -0.5684053 ,  0.06658808,  0.3106705 ,
        0.48493612, -0.4813773 , -0.7402723 , -0.00522922,  0.20

In [12]:
candidate_embeddings.shape

(7388, 128)

# Load item meta

In [13]:
metadata_df = pd.read_parquet(cfg.data.metadata_fp)
metadata_df

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Buy a Kindle,The Alibi,4.3,5532,"[In this suspenseful Southern thriller and #1,...","[Amazon.com Review, Sandra Brown's two previou...",7.99,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Sandra Brown (Author) Format: Kindle Edition,"[Books, Mystery, Thriller & Suspense, Thriller...","{""Publisher"": ""Grand Central Publishing (Augus...",B00BEK6ZR2,None,Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...
1,Buy a Kindle,Later,4.5,33092,"[“Part detective tale, part thriller…touching ...","[Review, #1 on The New York Times bestselling ...",9.99,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Stephen King (Author) Format: Kindle Edition,"[Books, Mystery, Thriller & Suspense, Thriller...","{""Publisher"": ""Hard Case Crime (March 2, 2021)...",B08F4GYM8W,None,Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...
2,Buy a Kindle,The Night Agent: A Novel,4.4,2731,[NOW ON NETFLIX! Starring Gabriel Basso and Lu...,"[Review, “, The Night Agent, is a whirlwind of...",8.99,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Matthew Quirk (Author) Format: Kindle Edition,"[Books, Mystery, Thriller & Suspense, Thriller...","{""Publisher"": ""William Morrow (January 15, 201...",B07B7LB9TN,None,Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...
3,Buy a Kindle,The Word Is Murder: A Novel (A Hawthorne and H...,4.3,15516,"[""One of the most entertaining mysteries of th...","[From the Back Cover, One bright spring mornin...",14.49,"{'hi_res': [None], 'large': ['https://m.media-...",{'title': ['The Word Is Murder: A Novel (Detec...,Anthony Horowitz (Author) Format: Kindle Edi...,"[Books, Mystery, Thriller & Suspense, Thriller...","{""Publisher"": ""Harper; Reprint edition (June 5...",B072PQXYYJ,None,Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...
4,Buy a Kindle,The Hard Way Home (The Star and the Shamrock B...,4.7,10964,[Dublin 1950Liesl Bannon has never felt like s...,[],0.0,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Jean Grainger (Author) Format: Kindle Edition,"[Books, Literature & Fiction, Genre Fiction]","{""Publication date"": ""June 29, 2020"", ""Languag...",B088HJ5312,None,Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7383,Books,Adult Coloring Book: Stress Relieving Patterns,4.5,4342,[STRESS RELIEVING | CALMING | RELAXING | CREAT...,[],6.91,"{'hi_res': [], 'large': [], 'thumb': [], 'vari...","{'title': [], 'url': [], 'user_id': []}","Blue Star Coloring (Author), Adult Coloring B...","[Books, Arts & Photography, History & Criticism]","{""Publisher"": ""Blue Star Coloring; Csm edition...",1941325122,None,"Paperback – March 28, 2015",{'avatar': 'https://m.media-amazon.com/images/...
7384,Books,Code Name Verity (Edgar Allen Poe Awards. Best...,4.3,5426,"[The beloved #1, New York Times, bestseller, a...","[Amazon.com Review, Amazon Best Teen Books of ...",9.19,"{'hi_res': [], 'large': [], 'thumb': [], 'vari...","{'title': [], 'url': [], 'user_id': []}",Elizabeth Wein (Author),"[Books, Teen & Young Adult, Literature & Fiction]","{""Publisher"": ""Little, Brown Books for Young R...",1423152190,None,"Hardcover – May 15, 2012",{'avatar': 'https://m.media-amazon.com/images/...
7385,Buy a Kindle,Cold Vengeance (Pendergast Book 11),4.6,3552,"[Twelve years ago, Special Agent Pendergast's ...","[Review, An exceptionally strong number of a b...",8.99,"{'hi_res': [], 'large': [], 'thumb': [], 'vari...","{'title': [], 'url': [], 'user_id': []}","Douglas Preston (Author), Lincoln Child (Auth...","[Books, Literature & Fiction, Genre Fiction]","{""Publisher"": ""Grand Central Publi

In [14]:
metadata_cols = [
    'main_category',
    'title',
    'average_rating',
    'rating_number',
    'price',
    cfg.data.item_col,
    'subtitle'
]
metadata_map = metadata_df[metadata_cols].set_index(cfg.data.item_col).to_dict(orient='index')
len(metadata_map)

7388

# Embedding store

In [15]:
ann_index = QdrantClient(url=cfg.vectorstore.qdrant.url)

In [16]:
collection_exists = ann_index.collection_exists(cfg.vectorstore.qdrant.collection_name)
if collection_exists:
    logger.info(f"Deleting existing Qdrant collection {cfg.vectorstore.qdrant.collection_name}...")
    ann_index.delete_collection(cfg.vectorstore.qdrant.collection_name)

logger.info(f"Creating Qdrant collection {cfg.vectorstore.qdrant.collection_name}...")
create_collection_result = ann_index.create_collection(
    collection_name=cfg.vectorstore.qdrant.collection_name,
    vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
)

assert create_collection_result == True

2025-03-11 20:15:00.910 | INFO     | __main__:<module>:3 - Deleting existing Qdrant collection seq_model_retrieve...
2025-03-11 20:15:00.916 | INFO     | __main__:<module>:6 - Creating Qdrant collection seq_model_retrieve...


In [17]:
points = []
for idx, vector in enumerate(candidate_embeddings):
    id_ = id_mapping.get_item_id(idx)
    payload = metadata_map[id_]
    point = PointStruct(id=idx, vector=vector.tolist(), payload=payload)
    points.append(point)

upsert_result = ann_index.upsert(
    collection_name=cfg.vectorstore.qdrant.collection_name,
    points=points,
)
assert str(upsert_result.status) == "completed"
upsert_result

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

# Sanity check

In [18]:
metadata_df.loc[metadata_df['title'].str.contains("(?i)harry potter", regex=True)]

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
117,Books,Harry Potter and the Order of the Phoenix (Boo...,4.8,75559,"[The next volume in the thrilling, moving, bes...","[Amazon.com Review, As his fifth year at Hogwa...",17.29,"{'hi_res': [None], 'large': ['https://m.media-...",{'title': ['Customer Review: wish I knew this ...,"J. K. Rowling (Author), Mary GrandPré (Illust...","[Books, Children's Books, Growing Up & Facts o...","{""Publisher"": ""Arthur A. Levine Books; First E...",043935806X,None,"Hardcover – July 1, 2003",{'avatar': 'https://m.media-amazon.com/images/...
139,Books,Harry Potter and the Prisoner of Azkaban (Harr...,4.9,80540,"[For twelve long years, the dread fortress of ...","[Amazon.com Review, For most children, summer ...",15.74,"{'hi_res': [None], 'large': ['https://m.media-...",{'title': ['Harry Potter Prisoner of Azkaban B...,"J.K. Rowling (Author), Mary GrandPré (Illustr...","[Books, Children's Books, Growing Up & Facts o...","{""Publisher"": ""Scholastic; First Edition (Octo...",0439136350,None,"Hardcover – October 1, 1999",{'avatar': 'https://m.media-amazon.com/images/...
257,Books,Harry Potter and the Sorcerer's Stone (1),4.9,8716,[Harry Potter spent ten long years living with...,"[Amazon.com Review, Say you've spent the first...",15.69,"{'hi_res': [None], 'large': ['https://m.media-...",{'title': ['Cutomer Review: Why should you get...,"J.K. Rowling (Author), Mary Grandpre (Illustr...","[Books, Children's Books, Growing Up & Facts o...","{""Publisher"": ""Scholastic Press (October 1, 19...",0590353403,None,"Hardcover – October 1, 1998",{'avatar': 'https://m.media-amazon.com/images/...
704,Books,Harry Potter and the Chamber of Secrets: The I...,4.9,14496,[Award-winning artist Jim Kay illustrates year...,"[Review, ""Seeing Jim Kay's illustrations moved...",23.63,"{'hi_res': [None], 'large': ['https://m.media-...",{'title': ['Check out what is inside this book...,"J. K. Rowling (Author), Mr. Jim Kay (Illustra...","[Books, Children's Books, Growing Up & Facts o...","{""Publisher"": ""Arthur A. Levine Books; Illustr...",0545791324,None,"Hardcover – Illustrated, October 4, 2016",{'avatar': 'https://m.media-amazon.com/images/...
1198,Books,"Harry Potter and the Cursed Child, Parts 1 & 2...",4.3,100137,"["", The Eighth Story. Nineteen Years Later., B...","[From School Library Journal, Playwright Thorn...",9.29,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}","J.K. Rowling (Author), Jack Thorne (Author), ...","[Books, Children's Books, Science Fiction & Fa...","{""Publisher"": ""Arthur A. Levine Books; Special...",1338099132,None,"Hardcover – July 31, 2016",{'avatar': 'https://m.media-amazon.com/images/...
1389,Books,Harry Potter Hardcover Boxed Set: Books 1-7,4.9,111418,[HARRY POTTER BOXED SET 1-7 includes the seven...,"[About the Author, J.K. ROWLING is the author ...",154.09,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': ['Perfect condition!', 'Great condit...",J. K. Rowling (Author),"[Books, Teen & Young Adult, Science Fiction & ...","{""Publisher"": ""Scholastic Inc.; First Thus edi...",0545044251,None,"Hardcover – Box set, October 16, 2007",{'avatar': 'https://m.media-amazon.com/images/...
1406,Books,Harry Potter and the Half-Blood Prince (Book 6),4.9,71339,[As the Harry Potter sequence draws to a close...,"[Amazon.com Review, The long-awaited, eagerly ...",16.11,"{'hi_res': [None], 'large': ['https://m.media-...",{'title': ['Harry Potter Half-Blood Prince Boo...,"J. K. Rowling (Author), Mary GrandPré (Illust...","[Books, Children's Books, Growing Up & Facts o...","{""Publisher"": ""Arthur A. Levine Books (August ...",0439784549,None,"Hardcover – August 1, 2005",{'avatar': 'https://m.media-amazon.com/images/...
1564,Books,Harry Potter and the Sorcerer's Stone: The Ill...,4.9,25381,[The beloved first book of the Harry Potter se...,

In [46]:
id_ = '043935806X'
idx = id_mapping.get_item_index(id_)
inputs = {
    "item_seq": torch.tensor([[idx]])
}
query_embedding = model.get_query_embeddings(inputs)[0]
logger.info(f"{query_embedding.shape=}")

2025-03-11 20:22:15.497 | INFO     | __main__:<module>:7 - query_embedding.shape=torch.Size([128])


In [47]:
hits = ann_index.search(
    collection_name=cfg.vectorstore.qdrant.collection_name,
    query_vector=query_embedding,
    limit=cfg.eval.top_k_retrieve,
)

/tmp/ipykernel_16404/3324227506.py:1: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = ann_index.search(


In [48]:
metadata_map[id_]

{'main_category': 'Books',
 'title': 'Harry Potter and the Order of the Phoenix (Book 5)',
 'average_rating': 4.8,
 'rating_number': 75559,
 'price': '17.29',
 'subtitle': 'Hardcover – July 1, 2003'}

In [49]:
hits

[ScoredPoint(id=1381, version=0, score=0.885266, payload={'main_category': 'Books', 'title': 'Harry Potter and the Chamber of Secrets', 'average_rating': 4.8, 'rating_number': 85813, 'price': '6.81', 'subtitle': 'Hardcover – Big Book, July 1, 1999'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=958, version=0, score=0.85394025, payload={'main_category': 'Books', 'title': 'Cutting for Stone: A novel', 'average_rating': 4.6, 'rating_number': 13593, 'price': '39.8', 'subtitle': 'Hardcover – Deckle Edge, February 3, 2009'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3162, version=0, score=0.8496342, payload={'main_category': 'Books', 'title': 'The Kite Runner', 'average_rating': 4.7, 'rating_number': 33187, 'price': '8.97', 'subtitle': 'Paperback – US Import, April 27, 2004'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1853, version=0, score=0.84823483, payload={'main_category': 'Books', 'title': 'A Feast for Crows (A Song of Ice